# 02 - Deep Learning Basics

**Goal:** Understand the building blocks of neural networks before we dive into Transformers.

We'll cover:
1. What is a neural network (neuron → layer → network)
2. Weights, biases, and activation functions
3. Loss functions (cross-entropy)
4. Gradient descent
5. Word embeddings
6. PyTorch basics
7. Train a model on XOR

**Prerequisites:** Notebook 01 (linear algebra essentials)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

np.random.seed(42)
torch.manual_seed(42)
print("Ready!")

## 1. What is a Neural Network?

### A Single Neuron

A neuron does three things:
1. **Multiply** each input by a weight
2. **Add** a bias
3. Apply an **activation function**

$$y = f(w_1 x_1 + w_2 x_2 + ... + w_n x_n + b)$$

Think of it as a tiny decision-maker: it looks at inputs, weighs their importance, and outputs a signal.

In [ ]:
# A single neuron from scratch
def neuron(inputs, weights, bias):
    """Single neuron: weighted sum + bias."""
    return np.dot(inputs, weights) + bias

# Example: is this email spam?
# inputs: [has_link, has_money_word, from_known_sender]
inputs = np.array([1.0, 1.0, 0.0])
weights = np.array([0.5, 0.8, -0.9])  # links and money words → spam; known sender → not spam
bias = -0.3

raw_output = neuron(inputs, weights, bias)
print(f"Inputs:  {inputs}")
print(f"Weights: {weights}")
print(f"Bias:    {bias}")
print(f"Output:  {raw_output:.2f}  (positive → likely spam)")

### From Neuron to Network

- **Layer**: multiple neurons side by side (each with its own weights)
- **Network**: stack layers so the output of one layer feeds into the next

```
Input → [Layer 1: 4 neurons] → [Layer 2: 3 neurons] → [Output: 1 neuron]
```

This is called a **feedforward** network — data flows in one direction.

In [ ]:
# A layer = matrix multiplication!
# 3 inputs → 4 neurons = weight matrix of shape (3, 4)
W1 = np.random.randn(3, 4) * 0.5
b1 = np.zeros(4)

inputs = np.array([1.0, 0.5, -0.3])
layer1_output = inputs @ W1 + b1

print(f"Input shape:  {inputs.shape}")
print(f"W1 shape:     {W1.shape}")
print(f"Output shape: {layer1_output.shape}")
print(f"Output:       {layer1_output}")
print()
print("Each of the 4 output values = one neuron's response!")

## 2. Activation Functions

Without activation functions, stacking layers would just be one big linear transformation (matrix multiply). Activation functions add **non-linearity**, letting networks learn complex patterns.

Two key ones:
- **Sigmoid**: squashes output to (0, 1) — good for probabilities
- **ReLU**: returns max(0, x) — simple and effective, used in Transformers

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def relu(x):
    return np.maximum(0, x)

x = np.linspace(-5, 5, 200)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(x, sigmoid(x), 'b-', linewidth=2)
axes[0].axhline(y=0, color='k', linewidth=0.5)
axes[0].axhline(y=0.5, color='gray', linewidth=0.5, linestyle='--')
axes[0].set_title('Sigmoid')
axes[0].set_xlabel('x')
axes[0].set_ylabel('sigmoid(x)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(x, relu(x), 'r-', linewidth=2)
axes[1].axhline(y=0, color='k', linewidth=0.5)
axes[1].set_title('ReLU')
axes[1].set_xlabel('x')
axes[1].set_ylabel('relu(x)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Sigmoid: smooth, squashes to (0,1). Used for output probabilities.")
print("ReLU: simple, fast. Used in hidden layers of Transformers.")

In [ ]:
# Two-layer network with ReLU
W1 = np.random.randn(3, 4) * 0.5
b1 = np.zeros(4)
W2 = np.random.randn(4, 1) * 0.5
b2 = np.zeros(1)

inputs = np.array([1.0, 0.5, -0.3])

# Forward pass
z1 = inputs @ W1 + b1        # linear
a1 = relu(z1)                 # activation
z2 = a1 @ W2 + b2             # linear
output = sigmoid(z2)          # final activation

print(f"Input:        {inputs}")
print(f"After layer1: {z1}")
print(f"After ReLU:   {a1}")
print(f"After layer2: {z2}")
print(f"Final output: {output}  (a probability!)")

## 3. Loss Functions

A **loss function** measures how wrong the network is. The goal of training is to **minimize the loss**.

### Cross-Entropy Loss

For classification: "how surprised are we by the prediction?"

$$\text{Loss} = -\sum_i y_i \log(\hat{y}_i)$$

- If true label is class 1 and we predict 0.99 → loss is tiny ✓
- If true label is class 1 and we predict 0.01 → loss is huge ✗

In [ ]:
def cross_entropy(y_true, y_pred):
    """Binary cross-entropy loss."""
    epsilon = 1e-15  # avoid log(0)
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# True label: 1 (spam)
predictions = np.linspace(0.01, 0.99, 100)
losses = [cross_entropy(1.0, p) for p in predictions]

plt.figure(figsize=(8, 4))
plt.plot(predictions, losses, 'b-', linewidth=2)
plt.xlabel('Predicted probability for correct class')
plt.ylabel('Cross-Entropy Loss')
plt.title('Loss decreases as prediction improves')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Predict 0.01 (wrong): loss = {cross_entropy(1.0, 0.01):.4f}")
print(f"Predict 0.50 (unsure): loss = {cross_entropy(1.0, 0.50):.4f}")
print(f"Predict 0.99 (right!): loss = {cross_entropy(1.0, 0.99):.4f}")

## 4. Gradient Descent

How does the network learn? **Gradient descent** — adjust weights to reduce the loss.

Analogy: You're on a foggy hill and want to reach the bottom. You can only feel the slope under your feet. Strategy: take a step in the **downhill direction**.

$$w_{\text{new}} = w_{\text{old}} - \alpha \cdot \frac{\partial \text{Loss}}{\partial w}$$

- $\alpha$ = **learning rate** (step size)
- $\frac{\partial \text{Loss}}{\partial w}$ = **gradient** (which direction is downhill)

In [ ]:
# 1D gradient descent demo
# Minimize f(x) = (x - 3)^2 + 1
def f(x):
    return (x - 3) ** 2 + 1

def df(x):
    return 2 * (x - 3)

# Start at x = -2, learning rate = 0.1
x = -2.0
lr = 0.1
history = [x]

for step in range(25):
    grad = df(x)
    x = x - lr * grad
    history.append(x)

# Plot
x_range = np.linspace(-3, 7, 200)
plt.figure(figsize=(10, 5))
plt.plot(x_range, f(x_range), 'b-', linewidth=2, label='f(x) = (x-3)² + 1')
plt.plot(history, [f(h) for h in history], 'ro-', markersize=6, label='Gradient descent steps')
plt.plot(history[0], f(history[0]), 'gs', markersize=12, label=f'Start (x={history[0]})')
plt.plot(history[-1], f(history[-1]), 'r*', markersize=15, label=f'End (x={history[-1]:.2f})')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.title('Gradient Descent: Rolling Downhill')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Started at x = {history[0]}, ended at x = {history[-1]:.4f} (optimal = 3.0)")

### Exercise 1: Learning Rate Experiment

Try changing the learning rate:
- `lr = 0.01` — what happens? (too slow?)
- `lr = 0.5` — what happens? (too fast?)
- `lr = 1.1` — what happens? (diverges!)

In [ ]:
# Exercise 1: Try different learning rates
# Your code here:

# Solution:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
x_range = np.linspace(-3, 9, 200)

for ax, lr in zip(axes, [0.01, 0.5, 1.1]):
    x = -2.0
    history = [x]
    for _ in range(25):
        x = x - lr * df(x)
        history.append(x)
    ax.plot(x_range, f(x_range), 'b-', linewidth=2)
    ax.plot(history[:15], [f(h) for h in history[:15]], 'ro-', markersize=4)
    ax.set_title(f'lr = {lr}')
    ax.set_ylim(-1, 30)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Too small → slow convergence. Too large → overshooting or divergence!")

## 5. Word Embeddings

Computers don't understand words — they need numbers. **Word embeddings** represent words as vectors.

Key ideas:
- Each word → a vector of, say, 4 numbers
- Similar words → similar vectors (close together)
- The famous result: king - man + woman ≈ queen

In Transformers, the **embedding dimension** (d_model) is typically 512.

In [ ]:
# Toy word embeddings (normally these are learned!)
embeddings = {
    'king':   np.array([0.9, 0.8, 0.1, 0.9]),
    'queen':  np.array([0.9, 0.8, 0.9, 0.9]),
    'man':    np.array([0.5, 0.6, 0.1, 0.3]),
    'woman':  np.array([0.5, 0.6, 0.9, 0.3]),
    'cat':    np.array([0.1, 0.2, 0.5, 0.1]),
    'dog':    np.array([0.2, 0.3, 0.5, 0.1]),
}

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Similarity matrix
words = list(embeddings.keys())
n = len(words)
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = cosine_sim(embeddings[words[i]], embeddings[words[j]])

plt.figure(figsize=(6, 5))
plt.imshow(sim_matrix, cmap='YlOrRd', vmin=0.5, vmax=1.0)
plt.xticks(range(n), words, rotation=45)
plt.yticks(range(n), words)
plt.colorbar(label='Cosine Similarity')
plt.title('Word Similarity (Cosine)')
for i in range(n):
    for j in range(n):
        plt.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center', fontsize=9)
plt.tight_layout()
plt.show()

# The famous analogy
result = embeddings['king'] - embeddings['man'] + embeddings['woman']
print("\nking - man + woman =", result)
print("queen              =", embeddings['queen'])
print(f"Similarity: {cosine_sim(result, embeddings['queen']):.4f}  (close to 1 = match!)")

## 6. PyTorch Basics

PyTorch is like NumPy but with two superpowers:
1. **GPU acceleration** — fast matrix math
2. **Automatic differentiation** — computes gradients for you!

The key abstraction: `torch.Tensor` (like `np.array` but tracks gradients).

In [ ]:
# Tensors — like numpy arrays
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print("a + b =", a + b)
print("a @ b =", a @ b)          # dot product
print("a.shape =", a.shape)

# From numpy
np_arr = np.array([[1, 2], [3, 4]])
t = torch.from_numpy(np_arr).float()
print("\nFrom numpy:", t)

# The magic: autograd!
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2 + 3 * x + 1  # y = x² + 3x + 1
y.backward()             # compute dy/dx
print(f"\ny = x² + 3x + 1 at x=2: y = {y.item():.1f}")
print(f"dy/dx = 2x + 3 at x=2: {x.grad.item():.1f} (should be 7.0)")

In [ ]:
# nn.Module — PyTorch's way to define neural networks
class SimpleNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.layer2 = nn.Linear(hidden_size, output_size)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.layer1(x)    # linear transform
        x = self.relu(x)      # activation
        x = self.layer2(x)    # linear transform
        return x

model = SimpleNet(3, 8, 1)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters())}")

# Forward pass
x = torch.tensor([1.0, 0.5, -0.3])
output = model(x)
print(f"Input:  {x}")
print(f"Output: {output.item():.4f}")

## 7. Full Training Loop: Learning XOR

XOR is the classic problem that a single neuron **cannot** solve (it's not linearly separable). But a 2-layer network can!

| x1 | x2 | XOR |
|----|----|-----|
| 0  | 0  |  0  |
| 0  | 1  |  1  |
| 1  | 0  |  1  |
| 1  | 1  |  0  |

This demonstrates the full training loop:
1. **Forward pass** — compute predictions
2. **Compute loss** — how wrong are we?
3. **Backward pass** — compute gradients
4. **Update weights** — gradient descent step

In [ ]:
# XOR dataset
X = torch.tensor([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=torch.float32)
y = torch.tensor([[0], [1], [1], [0]], dtype=torch.float32)

# Model
class XORNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2, 8)
        self.layer2 = nn.Linear(8, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.sigmoid(self.layer2(x))
        return x

model = XORNet()
loss_fn = nn.BCELoss()              # Binary cross-entropy
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training loop
losses = []
for epoch in range(2000):
    pred = model(X)                 # 1. Forward pass
    loss = loss_fn(pred, y)         # 2. Compute loss
    optimizer.zero_grad()           # Clear old gradients
    loss.backward()                 # 3. Backward pass
    optimizer.step()                # 4. Update weights
    losses.append(loss.item())
    if epoch % 400 == 0:
        print(f"Epoch {epoch:4d} | Loss: {loss.item():.4f}")

print(f"Epoch 2000 | Loss: {losses[-1]:.4f}")

In [ ]:
# Plot training loss
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('XOR Training: Loss Over Time')
plt.grid(True, alpha=0.3)
plt.show()

# Check predictions
with torch.no_grad():
    predictions = model(X)
    print("\nFinal predictions:")
    for i in range(4):
        pred_val = predictions[i].item()
        label = y[i].item()
        print(f"  {X[i].tolist()} → {pred_val:.4f} (target: {label:.0f}) {'✓' if abs(pred_val - label) < 0.2 else '✗'}")

### Exercise 2: Change the Network

What happens if you:
1. Use only 2 hidden neurons instead of 8? Can it still learn XOR?
2. Remove the hidden layer entirely (just `Linear(2, 1) + sigmoid`)? Why does it fail?

In [ ]:
# Exercise 2: Modify the network
# Your code here:

# Solution — single layer (will fail!):
class SingleLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(2, 1)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return self.sigmoid(self.linear(x))

model_fail = SingleLayer()
optimizer_fail = optim.Adam(model_fail.parameters(), lr=0.01)
for epoch in range(2000):
    pred = model_fail(X)
    loss = loss_fn(pred, y)
    optimizer_fail.zero_grad()
    loss.backward()
    optimizer_fail.step()

with torch.no_grad():
    preds = model_fail(X)
    print("Single layer predictions (should fail):")
    for i in range(4):
        print(f"  {X[i].tolist()} → {preds[i].item():.4f} (target: {y[i].item():.0f})")
    print("\nIt can't learn XOR! A single linear boundary can't separate the classes.")

### Exercise 3: Gradient Descent on a 2D Function

Use PyTorch's autograd to find the minimum of $f(x, y) = (x - 1)^2 + (y + 2)^2$.

The minimum should be at (1, -2).

In [ ]:
# Exercise 3
# Your code here:

# Solution:
x = torch.tensor(5.0, requires_grad=True)
y_var = torch.tensor(5.0, requires_grad=True)
lr = 0.1

trajectory = []
for step in range(50):
    f = (x - 1) ** 2 + (y_var + 2) ** 2
    f.backward()
    trajectory.append((x.item(), y_var.item(), f.item()))
    with torch.no_grad():
        x -= lr * x.grad
        y_var -= lr * y_var.grad
        x.grad.zero_()
        y_var.grad.zero_()

print(f"Final: x = {x.item():.4f}, y = {y_var.item():.4f} (target: 1.0, -2.0)")
print(f"Final loss: {trajectory[-1][2]:.6f}")

## Summary

| Concept | What it is | Role in Transformers |
|---------|-----------|---------------------|
| Neuron/Layer | Weighted sum + activation | Building block of FFN layers |
| ReLU | max(0, x) activation | Used in Transformer's feed-forward layers |
| Cross-Entropy | Loss for classification | Training objective for language models |
| Gradient Descent | Minimize loss by following gradients | How Transformers are trained (via Adam) |
| Embeddings | Words as vectors | Input representation (d_model = 512) |
| PyTorch | Tensors + autograd + nn.Module | Framework we'll use to build Transformers |

**Next up:** Notebook 03 — Why do we need Transformers? The journey from RNNs to attention.